# Reviewer 1 — Revision Experiments
### Version-Context Detection of Malicious Package Updates (Scientific Reports, ID 35109da0)

This notebook produces every number needed to answer **Reviewer 1's five major comments**.

| Section | Reviewer 1 comment | Produces |
|---|---|---|
| 0 | data audit (supports R1-2) | feature taxonomy, AST sentinel rate, provenance counts |
| 1 | **R1-1** version-context claim | candidate-only / predecessor-only / full / shuffled / wrong-predecessor |
| 2 | **R1-3** ecosystem confounding | ecosystem-only baseline, indicator ablation, balanced priors, npm<->PyPI |
| 3 | **R1-4** leakage | random vs package-grouped vs temporal CV, nested CV |
| 4 | **R1-5** FPR | reconciliation, per-ecosystem from ONE unified model, PR curves, prevalence table |
| 5 | R3-9 (bonus) | single machine-readable results artifact |

**Before running:** upload `features.csv` and `merged_dataset_usable.csv` (Files panel, left sidebar).

> Every headline number in the revised manuscript must come from `revision_results.json` produced at the end.

In [ ]:
!pip install lightgbm -q
import warnings, json, numpy as np, pandas as pd
warnings.filterwarnings("ignore")
from sklearn.model_selection import (StratifiedKFold, StratifiedGroupKFold,
                                     cross_val_predict, GroupKFold, GridSearchCV)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             confusion_matrix, average_precision_score,
                             roc_auc_score, precision_recall_curve)
from scipy import stats
import lightgbm as lgb
import matplotlib.pyplot as plt

SEED = 42
RESULTS = {}   # single machine-readable results artifact

def metrics(y, pred, proba=None):
    tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
    out = dict(f1=f1_score(y, pred), precision=precision_score(y, pred, zero_division=0),
               recall=recall_score(y, pred), fpr=fp/(fp+tn) if (fp+tn) else 0.0,
               tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp))
    if proba is not None:
        out["pr_auc"] = average_precision_score(y, proba)
        out["roc_auc"] = roc_auc_score(y, proba)
    return {k: (round(v, 4) if isinstance(v, float) else v) for k, v in out.items()}

def show(name, m):
    print(f"{name:42s} F1={m['f1']:.3f}  Rec={m['recall']:.3f}  "
          f"Prec={m['precision']:.3f}  FPR={m['fpr']:.3f}")

### Load the data

Run this cell. If the files are not already in the session it opens a file picker —
select **`features.csv`** and **`merged_dataset_usable.csv`** together.

*(Colab wipes uploaded files whenever the runtime restarts, so just re-run this cell if that happens.)*

In [ ]:
import os

NEEDED = ["features.csv", "merged_dataset_usable.csv"]
missing = [f for f in NEEDED if not os.path.exists(f)]

if missing:
    print("missing:", missing)
    try:
        from google.colab import files
        print("Select BOTH csv files in the dialog...")
        up = files.upload()
    except ImportError:
        raise SystemExit("Not on Colab — place the csv files next to this notebook.")

still = [f for f in NEEDED if not os.path.exists(f)]
if still:
    # tolerate renamed uploads such as 'features (1).csv'
    import glob, shutil
    for want in still:
        stem = want.replace(".csv", "")
        cand = [g for g in glob.glob("*.csv") if g.startswith(stem)]
        if cand:
            shutil.copy(cand[0], want)
            print(f"recovered {want} from {cand[0]}")

for f in NEEDED:
    print(("  OK  " if os.path.exists(f) else "  MISSING  "), f)

In [ ]:
df = pd.read_csv("features.csv")
df["eco_npm"] = (df.ecosystem == "npm").astype(int)
df["dt"] = pd.to_datetime(df.version_after_date, format="mixed", utc=True)

y = df.label.values
groups = df.package.values

print("rows:", len(df), "| unique packages:", df.package.nunique())
print(df.groupby(["ecosystem", "label"]).size())

---
## 0. Data audit

Three things Reviewer 1 (comment 2) and the other reviewers need documented before any model is trained.

### 0.1 Feature taxonomy — which features actually need the predecessor?

R1-1 asks to *"quantify the incremental contribution of each feature that genuinely requires predecessor access."*
Before measuring, the features must be split into two explicit groups. **This table goes into the revised Table 2.**

In [ ]:
CANDIDATE_ONLY = [c for c in df.columns if c.startswith("after_")] + ["entropy_after", "ast_nodes_after"]
NEEDS_PREDECESSOR = [c for c in df.columns if c.startswith("delta_")] + ["entropy_before"]

print("CANDIDATE-ONLY (computable from the candidate archive alone):")
for c in CANDIDATE_ONLY: print("   ", c)
print("\nREQUIRES PREDECESSOR (needs a second archive download):")
for c in NEEDS_PREDECESSOR: print("   ", c)

# (recorded again after the AST check below, once the final set is fixed)

### 0.2 AST feature validity check

Reviewer 2 (§10) warned: *"If sentinel or failure values dominate the feature, it should be recomputed or removed."*
This cell measures the parse-failure rate per ecosystem. **Report this number in the revised Section 4.2.**

In [ ]:
sent = (df.ast_nodes_after == -1)
print(f"ast_nodes_after == -1 (parse failure):  {sent.mean():.1%}  ({sent.sum()} / {len(df)})")
print(df.groupby("ecosystem").apply(lambda g: (g.ast_nodes_after == -1).mean()).round(4))
print("\nnon-sentinel value counts:")
print(df.loc[~sent, "ast_nodes_after"].describe())

RESULTS["ast_parse_failure_rate"] = {
    "overall": round(float(sent.mean()), 4),
    "by_ecosystem": df.groupby("ecosystem").apply(
        lambda g: round(float((g.ast_nodes_after == -1).mean()), 4)).to_dict()}

# If failure rate is high, the feature is a constant and must be dropped from the headline model.
AST_USABLE = sent.mean() < 0.5
print("\n>>> AST feature usable in headline model:", AST_USABLE)
if not AST_USABLE:
    CANDIDATE_ONLY = [c for c in CANDIDATE_ONLY if c != "ast_nodes_after"]
    print(">>> ast_nodes_after DROPPED from the feature set (dominated by sentinel values).")

RESULTS["feature_taxonomy"] = {"candidate_only": CANDIDATE_ONLY,
                               "requires_predecessor": NEEDS_PREDECESSOR}

### 0.3 Provenance counts — answers the Algorithm 1 vs Algorithm 2 contradiction

R1-2 and R3-2 both ask: *how was a removed archive downloaded?* The answer is in `download_status`.
Only pairs marked `both_available` can reach feature extraction. **These counts replace Figure 2.**

In [ ]:
m = pd.read_csv("merged_dataset_usable.csv")
print(m.download_status.value_counts(), "\n")
print("source datasets:", m.source_dataset.unique().tolist())

n_both = int((m.download_status == "both_available").sum())
n_est  = int((m.download_status == "before_estimated_removed_after").sum())
n_final_mal = int((df.label == 1).sum())

print(f"\nidentifiable pairs                : {len(m)}")
print(f"  both archives downloadable      : {n_both}   <- the ONLY ones used for features")
print(f"  candidate removed / before est. : {n_est}   <- excluded at feature-extraction stage")
print(f"final malicious rows in features  : {n_final_mal}")
print(f"UNEXPLAINED GAP                   : {n_both - n_final_mal}  <- must be explained in the paper")

RESULTS["provenance"] = {"identifiable_pairs": len(m), "both_available": n_both,
                         "before_estimated": n_est, "final_malicious": n_final_mal,
                         "gap": n_both - n_final_mal}

---
## 1. R1-1 — Does version context actually contribute?

> *"Add a fully featured candidate-only baseline containing candidate-version structural features with
> comparable feature capacity. Quantify the incremental contribution of each feature that genuinely
> requires predecessor access."*

Five conditions, **identical classifier, identical folds, package-grouped** (so the answer is not
contaminated by the leakage of R1-4):

1. **candidate-only** — no predecessor archive needed at all
2. **predecessor-only** — only the delta/before features
3. **full** — both groups (the paper's current best)
4. **shuffled predecessor** — predecessor features randomly permuted across rows
5. **wrong predecessor** — predecessor features taken from a *different package in the same ecosystem*

Conditions 4 and 5 are the decisive test. If **full ≈ shuffled**, the predecessor carries no information
and the title must change. If **full > shuffled**, version context is real and defensible.

In [ ]:
cv = StratifiedGroupKFold(5, shuffle=True, random_state=SEED)

def clf():
    return lgb.LGBMClassifier(n_estimators=300, random_state=SEED, verbose=-1)

def run(X, name, store=None):
    proba = cross_val_predict(clf(), X, y, cv=cv, groups=groups, method="predict_proba")[:, 1]
    pred = (proba >= 0.5).astype(int)
    mm = metrics(y, pred, proba)
    show(name, mm)
    if store is not None: RESULTS.setdefault(store, {})[name] = mm
    return mm, proba

rng = np.random.RandomState(SEED)

X_cand = df[CANDIDATE_ONLY]
X_pred = df[NEEDS_PREDECESSOR]
X_full = df[CANDIDATE_ONLY + NEEDS_PREDECESSOR]

X_shuf = X_full.copy()
perm = rng.permutation(len(df))
for c in NEEDS_PREDECESSOR:
    X_shuf[c] = X_shuf[c].values[perm]

# WRONG predecessor: donor row drawn from a DIFFERENT package in the SAME ecosystem.
# (Rolling by one position would often donate from the same package - not a valid control.)
X_wrong = X_full.copy()
for eco in df.ecosystem.unique():
    idx = df.index[df.ecosystem == eco].to_numpy()
    pkgs = df.loc[idx, "package"].values
    donors = np.empty(len(idx), dtype=int)
    for j in range(len(idx)):
        while True:
            d = rng.randint(len(idx))
            if pkgs[d] != pkgs[j]:
                donors[j] = idx[d]; break
    for c in NEEDS_PREDECESSOR:
        X_wrong.loc[idx, c] = df.loc[donors, c].values

print("Package-grouped CV, LightGBM, ecosystem indicator EXCLUDED\n" + "-"*72)
m_cand, _  = run(X_cand,  "1. candidate-only",            "R1_1_predecessor_value")
m_pred, _  = run(X_pred,  "2. predecessor-only",          "R1_1_predecessor_value")
m_full, _  = run(X_full,  "3. full (candidate + pred)",   "R1_1_predecessor_value")
m_shuf, _  = run(X_shuf,  "4. full w/ SHUFFLED pred",     "R1_1_predecessor_value")
m_wrng, _  = run(X_wrong, "5. full w/ WRONG pred",        "R1_1_predecessor_value")

print("-"*72)
print(f"Incremental value of the CORRECT predecessor : {m_full['f1'] - m_shuf['f1']:+.4f} F1")
print(f"Value of predecessor over candidate-only     : {m_full['f1'] - m_cand['f1']:+.4f} F1")

### 1.1 Per-feature contribution of the predecessor group (leave-one-out)

R1-1 asks to quantify **each** predecessor-requiring feature. Anything contributing < 0.005 F1
cannot justify a second archive download (Reviewer 4's cost argument).

In [ ]:
base = m_full["f1"]
rows = []
for c in NEEDS_PREDECESSOR:
    cols = [x for x in X_full.columns if x != c]
    p = cross_val_predict(clf(), df[cols], y, cv=cv, groups=groups)
    d = base - f1_score(y, p)
    rows.append({"feature": c, "F1_without": round(f1_score(y, p), 4), "drop": round(d, 4)})

loo = pd.DataFrame(rows).sort_values("drop", ascending=False)
print(f"full-model F1 = {base:.4f}\n")
print(loo.to_string(index=False))
RESULTS["R1_1_leave_one_out"] = loo.to_dict("records")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
names = ["candidate\nonly", "predecessor\nonly", "full", "shuffled\npredecessor", "wrong\npredecessor"]
vals = [m_cand["f1"], m_pred["f1"], m_full["f1"], m_shuf["f1"], m_wrng["f1"]]
bars = ax.bar(names, vals, color=["#888", "#888", "#2c7fb8", "#d95f02", "#d95f02"])
ax.set_ylabel("F1 (package-grouped CV)"); ax.set_ylim(0, 1)
ax.axhline(m_cand["f1"], ls="--", c="k", lw=.8)
for b, v in zip(bars, vals): ax.text(b.get_x()+b.get_width()/2, v+.02, f"{v:.3f}", ha="center")
ax.set_title("Does the correct predecessor add anything?")
plt.tight_layout(); plt.savefig("fig_R1_1_predecessor.png", dpi=200); plt.show()

---
## 2. R1-3 — Ecosystem indicator: label prior or real signal?

> *"A classifier using only the ecosystem label ... would obtain approximately 70.2% accuracy and
> 0.717 F1 ... Report an ecosystem-only baseline, remove the ecosystem indicator in an ablation,
> balance class priors within each ecosystem, and conduct train-on-npm/test-on-PyPI as well as
> train-on-PyPI/test-on-npm evaluations."*

All four sub-requests, in order.

In [ ]:
# (a) ecosystem-only baseline — predict npm => malicious
pred_eco = df.eco_npm.values
m_eco = metrics(y, pred_eco)
show("(a) ECOSYSTEM-ONLY baseline", m_eco)
print(f"     accuracy = {(pred_eco == y).mean():.3f}   <- Reviewer 1 predicted ~0.702 / F1 0.717")

# (b) indicator ablation
m_with, _ = run(df[CANDIDATE_ONLY + NEEDS_PREDECESSOR + ["eco_npm"]], "(b) full WITH eco indicator")
m_wout    = m_full
show("(b) full WITHOUT eco indicator", m_wout)
print(f"     indicator contributes {m_with['f1'] - m_wout['f1']:+.4f} F1")

RESULTS["R1_3_ecosystem"] = {"ecosystem_only_baseline": m_eco,
                             "with_indicator": m_with, "without_indicator": m_wout}

In [ ]:
# (c) balanced class priors within each ecosystem (downsample majority per ecosystem)
parts = []
rs = np.random.RandomState(SEED)
for eco, g_ in df.groupby("ecosystem"):
    n = g_.label.value_counts().min()
    parts.append(pd.concat([g_[g_.label == k].sample(n, random_state=SEED) for k in (0, 1)]))
bal = pd.concat(parts).reset_index(drop=True)
print("balanced subset:"); print(bal.groupby(["ecosystem", "label"]).size())

Xb = bal[CANDIDATE_ONLY + NEEDS_PREDECESSOR]
pb = cross_val_predict(clf(), Xb, bal.label, cv=StratifiedGroupKFold(5, shuffle=True, random_state=SEED),
                       groups=bal.package, method="predict_proba")[:, 1]
m_bal = metrics(bal.label, (pb >= .5).astype(int), pb)
show("(c) balanced priors within ecosystem", m_bal)
RESULTS["R1_3_ecosystem"]["balanced_priors"] = m_bal

In [ ]:
# (d) leave-one-ecosystem-out, BOTH directions -- the only real test of transfer
FEATS = CANDIDATE_ONLY + NEEDS_PREDECESSOR
loeo = {}
for src, tgt in [("npm", "pypi"), ("pypi", "npm")]:
    tr, te = df[df.ecosystem == src], df[df.ecosystem == tgt]
    mdl = clf().fit(tr[FEATS], tr.label)
    pr = mdl.predict_proba(te[FEATS])[:, 1]
    mm = metrics(te.label, (pr >= .5).astype(int), pr)
    show(f"(d) train {src} -> test {tgt}", mm)
    loeo[f"{src}_to_{tgt}"] = mm
RESULTS["R1_3_ecosystem"]["leave_one_ecosystem_out"] = loeo

print("\n>>> If both directions collapse, replace 'cross-ecosystem transfer'")
print(">>> with 'joint npm/PyPI training' throughout the manuscript.")

---
## 3. R1-4 — Leakage: random vs package-grouped vs temporal

> *"Add package-grouped splits, code-duplicate or family-level leakage checks, and a strict temporal
> evaluation in which all training data precede the test data."*

The dataset has up to 34 version pairs from a single package. Under random CV those pairs sit in both
the training and test folds — the classifier can memorise the package instead of learning the attack.

In [ ]:
print("rows per package (top 8):"); print(df.package.value_counts().head(8), "\n")
print("packages appearing under BOTH labels:", int(df.groupby('package').label.nunique().gt(1).sum()))

Xf = df[CANDIDATE_ONLY + NEEDS_PREDECESSOR + ["eco_npm"]]

pr_rand = cross_val_predict(clf(), Xf, y, cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
                            method="predict_proba")[:, 1]
m_rand = metrics(y, (pr_rand >= .5).astype(int), pr_rand)
show("random stratified CV  (paper's protocol)", m_rand)

pr_grp = cross_val_predict(clf(), Xf, y, cv=cv, groups=groups, method="predict_proba")[:, 1]
m_grp = metrics(y, (pr_grp >= .5).astype(int), pr_grp)
show("package-grouped CV    (leakage-free)", m_grp)

print(f"\n>>> LEAKAGE INFLATION: {m_rand['f1'] - m_grp['f1']:+.4f} F1")
RESULTS["R1_4_leakage"] = {"random_cv": m_rand, "package_grouped_cv": m_grp,
                           "inflation_f1": round(m_rand['f1'] - m_grp['f1'], 4)}

### 3.1 Payload-family leakage check

Attack campaigns publish the *same payload* under many package names. Deduplication on
`(ecosystem, package, version)` does not remove them. This clusters rows by their feature
fingerprint and re-runs the evaluation with the cluster as the grouping unit.

In [ ]:
sig_cols = [c for c in df.columns if c.startswith(("after_", "delta_"))]
sig = df[sig_cols].round(3).astype(str).agg("|".join, axis=1)
df["family"] = pd.factorize(sig)[0]

dup = df.family.value_counts()
print(f"distinct fingerprints: {df.family.nunique()} for {len(df)} rows")
print(f"rows in a repeated-payload cluster: {int(dup[dup > 1].sum())}")
print("largest clusters:"); print(dup.head(5))

pr_fam = cross_val_predict(clf(), Xf, y, cv=cv, groups=df.family, method="predict_proba")[:, 1]
m_fam = metrics(y, (pr_fam >= .5).astype(int), pr_fam)
show("payload-family-grouped CV", m_fam)
RESULTS["R1_4_leakage"]["family_grouped_cv"] = m_fam

### 3.2 Strict temporal evaluation

Figure 1 of the manuscript advertises a temporal split while Section 10 calls it future work.
**This cell resolves that contradiction** — run it, and either report the result or remove the
split from Figure 1.

In [ ]:
CUTOFF = "2026-01-01"     # tune so the test fold holds both classes
tr = df[df.dt < CUTOFF]; te = df[df.dt >= CUTOFF]
print(f"train (< {CUTOFF}): {len(tr)}  malicious={int(tr.label.sum())}")
print(f"test  (>={CUTOFF}): {len(te)}  malicious={int(te.label.sum())}")

if te.label.nunique() > 1 and tr.label.nunique() > 1:
    mdl = clf().fit(tr[FEATS + ["eco_npm"]], tr.label)
    pr = mdl.predict_proba(te[FEATS + ["eco_npm"]])[:, 1]
    m_tmp = metrics(te.label, (pr >= .5).astype(int), pr)
    show(f"temporal hold-out (cutoff {CUTOFF})", m_tmp)
    RESULTS["R1_4_leakage"]["temporal_holdout"] = {"cutoff": CUTOFF, **m_tmp}
else:
    print("Single class in a split -- move the cutoff.")

### 3.3 Nested CV — hyperparameter selection isolated from reporting

> *"It is unclear whether hyperparameter selection ... was isolated from the reported performance estimates."*

In [ ]:
outer = StratifiedGroupKFold(5, shuffle=True, random_state=SEED)
grid = {"n_estimators": [200, 400], "num_leaves": [15, 31], "learning_rate": [0.05, 0.1]}
scores = []
for k, (tr_i, te_i) in enumerate(outer.split(Xf, y, groups), 1):
    gs = GridSearchCV(clf(), grid, scoring="f1",
                      cv=GroupKFold(3).split(Xf.iloc[tr_i], y[tr_i], groups[tr_i]), n_jobs=-1)
    gs.fit(Xf.iloc[tr_i], y[tr_i])
    s = f1_score(y[te_i], gs.predict(Xf.iloc[te_i]))
    scores.append(s)
    print(f"  outer fold {k}: F1={s:.4f}   best={gs.best_params_}")

scores = np.array(scores)
ci = stats.t.interval(0.95, len(scores)-1, loc=scores.mean(), scale=stats.sem(scores))
print(f"\nNested CV F1 = {scores.mean():.4f}  95% CI [{ci[0]:.4f}, {ci[1]:.4f}]")
RESULTS["R1_4_leakage"]["nested_cv"] = {"fold_scores": [round(float(s), 4) for s in scores],
                                        "mean": round(float(scores.mean()), 4),
                                        "ci95": [round(float(ci[0]), 4), round(float(ci[1]), 4)]}

---
## 4. R1-5 — False-positive rate: reconciliation and honest deployment framing

> *"Section 6.3 reports a Random Forest false-positive rate of 0.073 ... the per-ecosystem table
> reports FPRs of 0.183 for npm and 0.013 for PyPI ... their weighted value is approximately 0.088,
> not 0.073. These values cannot all describe the same evaluation setting."*

The fix: derive per-ecosystem numbers from the **out-of-fold predictions of the one unified model**.
Then the weighted average reconciles by construction.

In [ ]:
proba_u = cross_val_predict(clf(), Xf, y, cv=cv, groups=groups, method="predict_proba")[:, 1]
pred_u = (proba_u >= .5).astype(int)

m_overall = metrics(y, pred_u, proba_u)
show("UNIFIED model, overall", m_overall)

per_eco, wfpr, wden = {}, 0.0, 0
for eco in ["npm", "pypi"]:
    msk = (df.ecosystem == eco).values
    mm = metrics(y[msk], pred_u[msk], proba_u[msk])
    # bootstrap CI for FPR (R1-5 asks for per-ecosystem uncertainty)
    b = []
    yb, pb_ = y[msk], pred_u[msk]
    for _ in range(2000):
        i = np.random.RandomState(SEED).choice(len(yb), len(yb), replace=True) if False else \
            np.random.choice(len(yb), len(yb), replace=True)
        tn, fp, fn, tp = confusion_matrix(yb[i], pb_[i], labels=[0, 1]).ravel()
        if fp + tn: b.append(fp/(fp+tn))
    mm["fpr_ci95"] = [round(float(np.percentile(b, 2.5)), 4), round(float(np.percentile(b, 97.5)), 4)]
    per_eco[eco] = mm
    show(f"  {eco}", mm)
    print(f"       FPR 95% CI {mm['fpr_ci95']}")
    wfpr += mm["fpr"] * (mm["fp"] + mm["tn"]); wden += mm["fp"] + mm["tn"]

print(f"\nweighted per-ecosystem FPR = {wfpr/wden:.4f}   overall FPR = {m_overall['fpr']:.4f}")
print(">>> These now agree because both come from the SAME unified model.")
RESULTS["R1_5_fpr"] = {"overall": m_overall, "per_ecosystem": per_eco,
                       "weighted_fpr": round(wfpr/wden, 4)}

### 4.1 Precision under realistic malicious prevalence

An FPR of ~8% is not "registry-scale ready" when malicious releases are rare. This table
belongs in the revised Section 6.3 and replaces the current deployment claim.

In [ ]:
R, F = m_overall["recall"], m_overall["fpr"]
rows = []
for p in [0.10, 0.05, 0.01, 0.005, 0.001, 0.0001]:
    prec = p*R / (p*R + (1-p)*F) if (p*R + (1-p)*F) else 0
    rows.append({"prevalence": f"{p:.2%}", "precision": round(prec, 4),
                 "false_alerts_per_10k": round((1-p)*F*10000, 1),
                 "true_alerts_per_10k": round(p*R*10000, 1)})
prev = pd.DataFrame(rows)
print(prev.to_string(index=False))
RESULTS["R1_5_fpr"]["prevalence_table"] = rows

In [ ]:
# Recall available at operationally realistic FPR budgets
tn_all = int((y == 0).sum())
rows = []
for target in [0.001, 0.005, 0.01, 0.05]:
    thr_grid = np.unique(proba_u)
    best = None
    for t in thr_grid:
        pr = (proba_u >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y, pr, labels=[0, 1]).ravel()
        f = fp/(fp+tn) if (fp+tn) else 0
        if f <= target:
            best = {"target_fpr": target, "threshold": round(float(t), 4),
                    "achieved_fpr": round(f, 4), "recall": round(tp/(tp+fn), 4)}
            break
    rows.append(best or {"target_fpr": target, "threshold": None,
                         "achieved_fpr": None, "recall": 0.0})
print(pd.DataFrame(rows).to_string(index=False))
RESULTS["R1_5_fpr"]["recall_at_fpr_budget"] = rows

In [ ]:
prec_c, rec_c, _ = precision_recall_curve(y, proba_u)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(rec_c, prec_c); ax[0].set_xlabel("Recall"); ax[0].set_ylabel("Precision")
ax[0].set_title(f"PR curve (PR-AUC = {m_overall['pr_auc']:.3f})"); ax[0].grid(alpha=.3)

ps = np.logspace(-4, -0.5, 100)
ax[1].plot(ps, ps*R/(ps*R + (1-ps)*F))
ax[1].set_xscale("log"); ax[1].set_xlabel("malicious prevalence"); ax[1].set_ylabel("precision")
ax[1].set_title("Precision vs prevalence"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig("fig_R1_5_prevalence.png", dpi=200); plt.show()

---
## 5. Single results artifact

Reviewer 3 (§9) requires that *"all tables, figures, and textual values"* be regenerated from
**one machine-readable source**. Download `revision_results.json` and cite every revised number from it.

In [ ]:
with open("revision_results.json", "w") as f:
    json.dump(RESULTS, f, indent=2, default=str)

print(json.dumps(RESULTS, indent=2, default=str)[:3000])
print("\n... saved to revision_results.json")

try:
    from google.colab import files
    files.download("revision_results.json")
except Exception:
    pass

---
## What to do with the output

| If you see | Then |
|---|---|
| full ≈ shuffled predecessor (§1) | version context is not the driver — retitle, and present the negative result as the contribution |
| full > shuffled by a clear margin (§1) | keep the central claim, but restate it at the **grouped-CV** numbers, not 0.895 |
| grouped F1 << random F1 (§3) | the 0.895 headline was leakage-inflated; every table and figure must be regenerated |
| both LOEO directions collapse (§2d) | delete "cross-ecosystem transfer"; write "joint npm/PyPI training" |
| AST failure rate > 50% (§0.2) | drop `ast_nodes_after` and say so in Section 4.2 |

Send me `revision_results.json` and I will rewrite the manuscript sections and draft the
point-by-point response to Reviewer 1.

## Primary size-matched controls (reproduces the 360/255 headline set)

Builds `primary_size_matched_dataset.csv` from the 360 malicious pairs and the 477 clean pool by without-replacement nearest-neighbour matching on candidate archive file count within ecosystem (ratio <= 3). 255 controls are admissible; all 360 positives are retained.

In [ ]:
# see code/build_primary_matched_controls.py for the standalone version
import numpy as np, pandas as pd, json
mal = pd.read_csv('features_v6_clean.csv', dtype=str); mal=mal[mal.label.astype(int)==1].copy()
pool = pd.read_csv('features_v6_controls.csv', dtype=str); pool['label']='0'
thr=np.log(3.0); pool=pool.copy(); pool['_u']=False; pick=[]
for _,m in mal.sample(frac=1,random_state=42).iterrows():
    cand=pool[(pool.ecosystem==m.ecosystem)&(~pool._u)]
    if not len(cand): continue
    d=(np.log1p(cand.files_total.astype(float))-np.log1p(float(m.files_total))).abs(); j=d.idxmin()
    if d.loc[j]>thr: continue
    pool.loc[j,'_u']=True; pick.append(j)
controls=pool.loc[pick].drop(columns='_u')
ds=pd.concat([mal,controls],ignore_index=True); ds['label']=ds.label.astype(int)
print('primary set:', len(ds), 'malicious', (ds.label==1).sum(), 'controls', (ds.label==0).sum())
ds.to_csv('primary_size_matched_dataset.csv', index=False)

## Version context on the primary pairs (true predecessor features)

Merges the true predecessor features (entropy_before, ast_nodes_before, delta_*) from the v4 table onto the primary malicious pairs and measures the drop when the predecessor is shuffled or drawn from a wrong package. Full logic in `code/version_context_experiment.py`.

In [ ]:
# summarised; full experiment in code/version_context_experiment.py
import json
print(json.dumps(json.load(open('version_context_primary_final.json'))['package_grouped'], indent=2))
# expected: correct ~0.718, shuffled ~0.674, gain +0.044

## Calibration and low-budget operating points (Reviewer 3)

Reliability diagram, Brier score, ECE, and operating points down to 0.1%/0.5% FPR. Full logic produces `calibration_and_operating_points.json`.

In [ ]:
from sklearn.metrics import brier_score_loss
import numpy as np, json
# oof probabilities `p` and labels `y` come from the pooled 3-seed size-matched evaluation above
# Brier and ECE:
# brier = brier_score_loss(y, p)
# ece computed over 10 equal-width bins of predicted probability
res=json.load(open('calibration_and_operating_points.json'))
print('Brier', res['calibration']['brier'], 'ECE', res['calibration']['ece'])
for o in res['operating_points_extended']:
    print(f"FPR<={o['fpr_budget']:.1%}: recall {o['recall']}, precision {o['precision']}")